# SAR-to-EO Image Translation — GalaxEye Assignment
**Model:** Pix2Pix U-Net + PatchGAN  
**Dataset:** Sentinel-1&2 terrain-split (Kaggle)

### ⚠️ Session Checklist
1. **Settings → Accelerator → GPU T4**
2. **Add dataset:** `requiemonk/sentinel12-image-pairs-segregated-by-terrain`
3. *(If resuming)* **Add previous session output** as Input dataset
4. Run **Cell 1 → 2 → 3 → 4**
5. Run the training cell(s) for this session
6. When done: run **Cell SAVE**, then **Stop Session** (power button)


In [ ]:
# ================================================================
# CELL 1: Environment check
# ================================================================
import subprocess, torch, os
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ================================================================
# CELL 2: Install packages
# ================================================================
!pip install lpips pytorch-fid scikit-image pyyaml -q
print('Packages installed.')

In [ ]:
# ================================================================
# CELL 3: Clone repo and set up working directory
# ================================================================
import os, shutil, subprocess

REPO_DIR = '/kaggle/working/sar2eo'
REPO_URL = 'https://github.com/Trafalgar-2006/sar2eo.git'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# ================================================================
# CELL 4: Find dataset + restore checkpoints from previous session
# ================================================================
import os, shutil, glob
from pathlib import Path

# --- Find Sentinel dataset ---
DATASET_ROOT = None
for base in Path('/kaggle/input').iterdir():
    for candidate in [base, *base.iterdir() if base.is_dir() else []]:
        if Path(candidate).is_dir() and any(Path(candidate).iterdir()):
            subdirs = [d.name.lower() for d in Path(candidate).iterdir() if d.is_dir()]
            if any(t in subdirs for t in ['agri', 'urban', 'grassland', 'barrenland']):
                DATASET_ROOT = str(candidate)
                break
    if DATASET_ROOT:
        break

assert DATASET_ROOT, 'Dataset not found! Add sentinel12 dataset to inputs.'
print(f'Dataset: {DATASET_ROOT}')

# --- Restore checkpoints from previous session (if mounted) ---
restored = 0
for zip_file in sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True)):
    if 'ckpt' in zip_file.lower() or 'checkpoint' in zip_file.lower():
        print(f'Restoring from: {zip_file}')
        shutil.unpack_archive(zip_file, '/kaggle/working/sar2eo')
        restored += 1

# Also check for .pth files directly mounted
for pth in sorted(glob.glob('/kaggle/input/**/*.pth', recursive=True)):
    ablation = Path(pth).parent.name
    dest_dir = f'/kaggle/working/sar2eo/checkpoints/{ablation}'
    os.makedirs(dest_dir, exist_ok=True)
    dest = f'{dest_dir}/{Path(pth).name}'
    if not os.path.exists(dest):
        shutil.copy2(pth, dest)
        print(f'  Copied: {Path(pth).name} → {ablation}/')
        restored += 1

if restored == 0:
    print('No previous checkpoints found — starting from scratch.')

# List restored checkpoints
for ckpt in sorted(glob.glob('/kaggle/working/sar2eo/checkpoints/**/*.pth', recursive=True)):
    sz = os.path.getsize(ckpt) / 1e6
    print(f'  Checkpoint: {ckpt.replace("/kaggle/working/sar2eo/", "")} ({sz:.1f} MB)')

In [ ]:
# ================================================================
# CELL 5: Configure paths
# ================================================================
import yaml, os

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['dataset_type']    = 'kaggle'
cfg['data']['kaggle_root']     = DATASET_ROOT
cfg['data']['train_terrain']   = ['agri', 'barrenland', 'grassland']
cfg['data']['val_terrain']     = ['urban']
cfg['data']['test_terrain']    = ['urban']
cfg['data']['subset_size']     = None
cfg['data']['num_workers']     = 2

cfg['training']['batch_size']  = 4
cfg['training']['epochs']      = 100

cfg['paths']['checkpoint_dir'] = '/kaggle/working/sar2eo/checkpoints'
cfg['paths']['output_dir']     = '/kaggle/working/sar2eo/outputs'

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config set:')
print(f'  dataset     : {DATASET_ROOT}')
print(f'  checkpoints : /kaggle/working/sar2eo/checkpoints')
print(f'  batch_size  : {cfg["training"]["batch_size"]}')
print(f'  epochs      : {cfg["training"]["epochs"]}')
print(f'  num_workers : {cfg["data"]["num_workers"]}')

In [ ]:
# ================================================================
# CELL 6: Smoke test (run ONCE to verify everything works)
# Skip this cell when resuming — jump straight to training
# ================================================================
import yaml, subprocess, copy, os

with open('config.yaml') as f:
    cfg_s = yaml.safe_load(f)

cfg_s['data']['subset_size']     = 100
cfg_s['training']['epochs']      = 2
cfg_s['training']['val_freq']    = 1
cfg_s['training']['save_freq']   = 2
cfg_s['active_ablation']         = 'full'
cfg_s['paths']['checkpoint_dir'] = '/kaggle/working/smoke_ckpts'
cfg_s['paths']['output_dir']     = '/kaggle/working/smoke_outputs'

with open('config_smoke.yaml', 'w') as f:
    yaml.dump(cfg_s, f)

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('Running smoke test (2 epochs, 100 pairs)...')
r = subprocess.run(
    ['python', '-u', 'train.py', '--config', 'config_smoke.yaml', '--ablation', 'full'],
    env=env
)
if r.returncode != 0:
    raise RuntimeError('Smoke test FAILED — check errors above.')
print('\nSmoke test PASSED ✅  Ready for full training.')

In [ ]:
# ================================================================
# CELL 7a: Config A — L1 only (baseline, ~3.7 hrs)
# Auto-resumes if checkpoint exists.
# ================================================================
import subprocess, os

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('Training Config A: L1 only...')
r = subprocess.run(
    ['python', '-u', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_only'],
    env=env
)
print('\nConfig A', 'DONE ✅' if r.returncode == 0 else 'FAILED ❌')

In [ ]:
# ================================================================
# CELL 7b: Config B — L1 + Adversarial (~6.3 hrs)
# Auto-resumes if checkpoint exists.
# ================================================================
import subprocess, os

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('Training Config B: L1 + Adversarial...')
r = subprocess.run(
    ['python', '-u', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv'],
    env=env
)
print('\nConfig B', 'DONE ✅' if r.returncode == 0 else 'FAILED ❌')

In [ ]:
# ================================================================
# CELL 7c: Config C — L1 + Adversarial + FFT (~6.3 hrs)
# Auto-resumes if checkpoint exists.
# ================================================================
import subprocess, os

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('Training Config C: L1 + Adversarial + FFT...')
r = subprocess.run(
    ['python', '-u', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv_fft'],
    env=env
)
print('\nConfig C', 'DONE ✅' if r.returncode == 0 else 'FAILED ❌')

In [ ]:
# ================================================================
# CELL 7d: Config D — Full model / MAIN SUBMISSION (~6.5 hrs)
# Auto-resumes if checkpoint exists.
# ================================================================
import subprocess, os

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('Training Config D (MAIN): Full loss stack...')
r = subprocess.run(
    ['python', '-u', 'train.py', '--config', 'config.yaml', '--ablation', 'full'],
    env=env
)
print('\nConfig D', 'DONE ✅' if r.returncode == 0 else 'FAILED ❌')

In [ ]:
# ================================================================
# CELL SAVE: Run this after training, then STOP SESSION
#
# This zips all checkpoints to /kaggle/working/
# When you Stop Session, Kaggle saves /kaggle/working/ as output.
# Mount that output in the next session to resume.
# ================================================================
import shutil, os, glob

ckpt_dir = '/kaggle/working/sar2eo/checkpoints'

# List what we have
print('=== Checkpoints to save ===')
for f in sorted(glob.glob(f'{ckpt_dir}/**/*.pth', recursive=True)):
    sz = os.path.getsize(f) / 1e6
    print(f'  {f.replace(ckpt_dir+"/", "")} ({sz:.1f} MB)')

# Zip the entire checkpoints folder
zip_out = '/kaggle/working/sar2eo_checkpoints'
shutil.make_archive(zip_out, 'zip', '/kaggle/working/sar2eo', 'checkpoints')
zip_size = os.path.getsize(f'{zip_out}.zip') / 1e6

print(f'\n✅ Zip created: sar2eo_checkpoints.zip ({zip_size:.1f} MB)')
print('\n🛑 NOW: Click the POWER BUTTON → Stop Session')
print('   Kaggle will save /kaggle/working/ as this session output.')
print('   Mount that output in the next session to resume training.')

In [ ]:
# ================================================================
# CELL 8: Evaluate all configs (run after all training is done)
# ================================================================
import subprocess, os

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

for ablation in ['l1_only', 'l1_adv', 'l1_adv_fft', 'full']:
    ckpt = f'/kaggle/working/sar2eo/checkpoints/{ablation}/best.pth'
    if not os.path.exists(ckpt):
        print(f'Skip {ablation} — no best.pth found')
        continue
    print(f'\n=== Evaluating {ablation} ===')
    subprocess.run([
        'python', '-u', 'eval.py',
        '--config', 'config.yaml',
        '--weights', ckpt,
        '--split', 'test'
    ], env=env)

print('\nAll evaluations done!')